# Projecte: BICITIERRA - Baseline

Ara se suposa que l'alumnat ja té clar els objectius del projecte i ha fet una primera exploració de les dades. En este quadern l'objectiu és crear baselines (models de referència, sense utilitzar algorismes de ML) per a la predicció de la variable objectiu. De moment, només en el dataset de vendes.

Els resultats de les prediccions que facen ací seran utilitzats com a referència per a comparar amb els resultats que s'obtinguen amb els models de Machine Learning.

## Ruta de treball

1. Triar variable objectiu.
2. Crear una baseline comprensible.
3. Comparar-la amb els valors reals.
4. Escriure què milloraries després.

In [185]:
# En primer lloc anem a carregar les llibreries necessàries i llegir el dataset de vendes.
# tin en compte que la ruta el dataset por variar en funció de com tingues organitzada la carpeta del projecte
# mostrem les 10 primeres files per una primera inspecció visual

import pandas as pd

df = pd.read_csv('../data/originales/sales_history.csv')
df.head(10)

,month,channel,product_line,region,marketing_spend,avg_discount,stock_units,website_sessions,sales_units,revenue,marketing_budget,planned_discount,list_price,campaign,days_in_month,closing_stock_units,stockout
0,2021-01,store,urban,coast,2190.14,0.047,150,3763.0,95,38024.70,2002.85,0.034,420.0,none,31,55,0
1,2021-01,store,urban,inland,3281.92,0.074,88,4664.0,82,31891.44,3016.71,0.062,420.0,none,31,6,0
2,2021-01,store,urban,metro,3347.16,0.050,141,5073.0,141,56259.00,3386.79,0.046,420.0,none,31,0,1
3,2021-01,store,mountain,coast,2530.35,0.037,66,3705.0,56,37210.32,2396.99,0.028,690.0,none,31,10,0
4,2021-01,store,mountain,inland,2136.16,0.101,61,3318.0,44,27293.64,2272.67,0.092,690.0,none,31,17,0
5,2021-01,store,mountain,metro,3566.49,0.011,48,4541.0,48,32755.68,3308.59,0.012,690.0,none,31,0,1
6,2021-01,store,electric,coast,1706.78,0.075,57,2693.0,39,53391.00,1811.32,0.071,1480.0,none,31,18,0
7,2021-01,store,electric,inland,3840.56,0.056,17,4483.0,17,23751.04,3652.61,0.047,1480.0,none,31,0,1
8,2021-01,store,electric,metro,2578.57,0.023,55,3765.0,48,69406.08,2722.88,0.043,1480.0,none,31,7,0
9,2021-01,ecommerce,urban,coast,3067.64,0.045,189,7085.0,130,52143.00,3138.31,0.046,420.0,none,31,59,0


### Exemple: Baseline per mitjana de grup

Com a aproximació simple, predim `sales_units` amb la mitjana del grup `channel + product_line`.

In [186]:
# calculem la mitjana mensual de vendes per cada combinació de `channel` i `product_line`
# dona-li un nom al resultat, per exemple "baseline_sales_units"
# i la guardem en una agrupació que podem anomenar `group_mean` 
df_grouped = df.groupby(['channel', 'product_line'])['sales_units'].mean().reset_index()
group_mean = df_grouped.rename(columns={'sales_units': 'baseline_sales_units'})

# investiga com adjuntar 2 dataframes utilitzant el mètode merge
# fes un merge per afegir la nova columna al dataframe original
# tenint en compte que en cada fila se posarà la mitjana corresponent al canal i el producte
# que corresponga
df = df.merge(group_mean, on=['channel', 'product_line'], how='left')

# ara podem mostrar el dataframe fent que apareguen el mes, el canal, el producte, les unitats
# venudes reals (sales_units) i la predicció (baseline_sales_units)
# mostra per exemple les primeres 15 files i fes una primera exploració visual
# per veure si hi ha molta diferència entre la predicció i la realitat
df[['month', 'channel', 'product_line', 'sales_units', 'baseline_sales_units']].head(15)

,month,channel,product_line,sales_units,baseline_sales_units
0,2021-01,store,urban,95,128.303030
1,2021-01,store,urban,82,128.303030
2,2021-01,store,urban,141,128.303030
3,2021-01,store,mountain,56,87.863636
4,2021-01,store,mountain,44,87.863636
5,2021-01,store,mountain,48,87.863636
6,2021-01,store,electric,39,55.454545
7,2021-01,store,electric,17,55.454545
8,2021-01,store,electric,48,55.454545
9,2021-01,ecommerce,urban,130,156.696970


In [187]:
# anem a veure la mitjana de l'error absolut (MAE) de la baseline
# l'error absolut es simplement el valor absolut de la diferencia entre el valor real i la predicció
# podeu crear una columna nova amb eixa informació
df['abs_error'] = (df['sales_units'] - df['baseline_sales_units']).abs()

# torneu a mostrar les primeres 15 files i que aparega també el MAE
df[['month', 'channel', 'product_line', 'sales_units', 'baseline_sales_units', 'abs_error']].head(15)

,month,channel,product_line,sales_units,baseline_sales_units,abs_error
0,2021-01,store,urban,95,128.303030,33.303030
1,2021-01,store,urban,82,128.303030,46.303030
2,2021-01,store,urban,141,128.303030,12.696970
3,2021-01,store,mountain,56,87.863636,31.863636
4,2021-01,store,mountain,44,87.863636,43.863636
5,2021-01,store,mountain,48,87.863636,39.863636
6,2021-01,store,electric,39,55.454545,16.454545
7,2021-01,store,electric,17,55.454545,38.454545
8,2021-01,store,electric,48,55.454545,7.454545
9,2021-01,ecommerce,urban,130,156.696970,26.696970


In [188]:
# podeu calcular l'error mig calculant la mitjana d'eixa columna nova que hem creat
# feu-ho i mostreu el resultat
mae = df['abs_error'].mean()
mae

np.float64(25.440507771315854)

In [189]:
# mostreu l'error en forma de percentatge sobre la totalitat de les vendes
# eixe valor guardeu-lo perquè després ens servirà per veure si la baseline
# millora o empitjora segons anem canviant coses
error_pct = 100 * df['abs_error'].sum() / df['sales_units'].sum()
error_pct

np.float64(25.378556748948892)

Eixe valor que acabeu de mostrar és el que vos ha de servir de referència amb la resta de baselines que feu, aplicant tots els canvis qeu se vos ocurrisquen. El repte és baixar eixe percentatge d'error.

In [190]:
# Per exemple, podem repetir tot el procés anterior provant a afegir la regió al grup i veure què passa.
group_mean_region = (
    df.groupby(['channel', 'product_line', 'region'])['sales_units']
      .mean()
      .reset_index()
      .rename(columns={'sales_units': 'baseline_region'})
)
df = df.merge(group_mean_region, on=['channel', 'product_line', 'region'], how='left')
df['abs_error_region'] = (df['sales_units'] - df['baseline_region']).abs()

mae_region = df['abs_error_region'].mean()
error_pct_region = 100 * df['abs_error_region'].sum() / df['sales_units'].sum()
print(f'MAE: {mae_region:.2f} | Error %: {error_pct_region:.2f} (abans: {mae:.2f} | {error_pct:.2f})')

MAE: 19.46 | Error %: 19.41 (abans: 25.44 | 25.38)


In [191]:
# Si no millora, treiem del grup la regió i afegim en el seu lloc la campanya 
# (columna campaign: ninguna, Nadal, Black Friday...)
group_mean_campaign = (
    df.groupby(['channel', 'product_line', 'campaign'])['sales_units']
      .mean()
      .reset_index()
      .rename(columns={'sales_units': 'baseline_campaign'})
)
# si la cel·la s'executa dos voltes, traiem la columna anterior perquè el merge no la duplique
df = df.drop(columns='baseline_campaign', errors='ignore')
df = df.merge(group_mean_campaign, on=['channel', 'product_line', 'campaign'], how='left')
df['abs_error_campaign'] = (df['sales_units'] - df['baseline_campaign']).abs()

mae_campaign = df['abs_error_campaign'].mean()
error_pct_campaign = 100 * df['abs_error_campaign'].sum() / df['sales_units'].sum()
print(f'MAE: {mae_campaign:.2f} | Error %: {error_pct_campaign:.2f}')

MAE: 23.65 | Error %: 23.59


In [192]:
# Una altra opció més complicada però que segurament donarà millors resultats és utilitzar
# la mitjana de les vendes només dels primers anys (per exemple 2021 a 2024) per a predir les vendes
# de 2025, i després comparar amb els valors reals (que els tenim)
train = df[df['month'].str[:4] <= '2024']
test = df[df['month'].str[:4] == '2025'].copy()

mean_train = (
    train.groupby(['channel', 'product_line', 'region'])['sales_units']
         .mean()
         .reset_index()
         .rename(columns={'sales_units': 'baseline_2021_2024'})
)
test = test.merge(mean_train, on=['channel', 'product_line', 'region'], how='left')
test['abs_error_2025'] = (test['sales_units'] - test['baseline_2021_2024']).abs()

mae_2025 = test['abs_error_2025'].mean()
error_pct_2025 = 100 * test['abs_error_2025'].sum() / test['sales_units'].sum()
print(f'MAE 2025: {mae_2025:.2f} | Error % 2025: {error_pct_2025:.2f}')

MAE 2025: 20.28 | Error % 2025: 19.54


### On falla més la baseline

Açò és útil perquè mostra si la simple agrupació ja captura prou bé el negoci o si encara queda molta variabilitat.

In [193]:
# Ordeneu per error absolut, de major a menor, i mostreu les 25 primeres files
df.sort_values(by='abs_error', ascending=False).head(25)



,month,channel,product_line,region,marketing_spend,avg_discount,stock_units,website_sessions,sales_units,revenue,...,campaign,days_in_month,closing_stock_units,stockout,baseline_sales_units,abs_error,baseline_region,abs_error_region,baseline_campaign,abs_error_campaign
1145,2026-04,ecommerce,urban,metro,3136.90,0.151,356,8775.0,328,128442.43,...,spring,30,28,0,156.696970,171.303030,187.893939,140.106061,191.750000,136.250000
986,2025-07,ecommerce,mountain,metro,3517.53,0.125,254,8437.0,229,149816.38,...,summer,31,25,0,104.606061,124.393939,128.000000,101.000000,150.800000,78.200000
968,2025-06,ecommerce,mountain,metro,3889.20,0.072,249,8478.0,228,157962.27,...,none,30,21,0,104.606061,123.393939,128.000000,100.000000,100.824675,127.175325
767,2024-07,ecommerce,urban,metro,2024.23,0.156,315,6978.0,271,102253.35,...,summer,31,44,0,156.696970,114.303030,187.893939,83.106061,199.250000,71.750000
272,2022-04,store,urban,metro,2707.09,0.139,234,5352.0,234,86527.04,...,spring,30,0,1,128.303030,105.696970,155.121212,78.878788,172.000000,62.000000
932,2025-04,ecommerce,mountain,metro,4014.25,0.127,210,8432.0,210,136461.69,...,spring,30,0,1,104.606061,105.393939,128.000000,82.000000,142.000000,68.000000
1148,2026-04,ecommerce,mountain,metro,3159.95,0.119,263,7505.0,203,135518.28,...,spring,30,60,0,104.606061,98.393939,128.000000,75.000000,142.000000,61.000000
1184,2026-06,ecommerce,mountain,metro,3565.79,0.047,198,8053.0,198,143407.44,...,none,30,0,1,104.606061,93.393939,128.000000,70.000000,100.824675,97.175325
113,2021-07,store,mountain,metro,5181.28,0.151,181,6842.0,181,106981.28,...,summer,31,0,1,87.863636,93.136364,106.636364,74.363636,128.571429,52.428571
983,2025-07,ecommerce,urban,metro,3113.89,0.145,295,7769.0,249,96890.64,...,summer,31,46,0,156.696970,92.303030,187.893939,61.106061,199.250000,49.750000


In [194]:
# Ordeneu per percentatge d'error, de major a menor, i mostreu les 25 primeres files
df['percent_error'] = (df['abs_error'] / df['sales_units']) * 100

df.sort_values(by='percent_error', ascending=False).head(25)

,month,channel,product_line,region,marketing_spend,avg_discount,stock_units,website_sessions,sales_units,revenue,...,days_in_month,closing_stock_units,stockout,baseline_sales_units,abs_error,baseline_region,abs_error_region,baseline_campaign,abs_error_campaign,percent_error
1048,2025-11,store,mountain,inland,1527.18,0.019,58,2778.0,25,18446.23,...,30,33,0,87.863636,62.863636,64.469697,39.469697,84.344371,59.344371,251.454545
7,2021-01,store,electric,inland,3840.56,0.056,17,4483.0,17,23751.04,...,31,0,1,55.454545,38.454545,40.500000,23.500000,53.550336,36.550336,226.203209
34,2021-02,ecommerce,electric,inland,2692.01,0.063,23,6037.0,23,31942.89,...,28,0,1,68.540404,45.540404,51.060606,28.060606,66.356164,43.356164,198.001757
241,2022-02,store,electric,inland,2912.53,0.071,33,3849.0,19,26633.24,...,28,14,0,55.454545,36.454545,40.500000,21.500000,53.550336,34.550336,191.866029
625,2023-11,ecommerce,mountain,inland,2035.55,0.161,38,5500.0,38,23139.00,...,30,0,1,104.606061,66.606061,75.287879,37.287879,80.900000,42.900000,175.279107
247,2022-02,ecommerce,mountain,inland,4234.15,0.052,38,7195.0,38,25341.80,...,28,0,1,104.606061,66.606061,75.287879,37.287879,100.824675,62.824675,175.279107
25,2021-02,store,electric,inland,2336.44,0.076,37,3153.0,21,28760.61,...,28,16,0,55.454545,34.454545,40.500000,19.500000,53.550336,32.550336,164.069264
655,2024-01,store,electric,inland,3237.58,0.051,34,3967.0,22,32598.28,...,31,12,0,55.454545,33.454545,40.500000,18.500000,53.550336,31.550336,152.066116
661,2024-01,ecommerce,mountain,inland,3759.57,0.062,42,6623.0,42,28677.53,...,31,0,1,104.606061,62.606061,75.287879,33.287879,100.824675,58.824675,149.062049
883,2025-02,store,urban,inland,NaN,0.065,53,3169.0,53,22385.98,...,28,0,1,128.303030,75.303030,93.348485,40.348485,121.954545,68.954545,142.081189


En els llistats anteriors podreu veure coses com si els errors més grans se concentren en un canal, en un producte, en una regió, en un periode, etc.

In [195]:
# Com que ja sabem fer agrupacions, i ja tenim una columna nova amb l'error absolut
# podem agrupar per canal i mostrar l'error mig
#
# Creeu nous blocs de codi per a mostrar la mitjana de l'error seguint diferents agrupacions: 
# per producte, per regió, per producte i canal, per canal i campanya... Qualsevol combinació 
# que vos vinga al cap i que tinga sentit

df.groupby('channel')['abs_error'].mean().reset_index().sort_values(by='abs_error', ascending=False)

,channel,abs_error
0,ecommerce,27.506666
1,store,23.374350


In [196]:
df.groupby(['product_line'])['abs_error'].mean().reset_index().sort_values(by='abs_error', ascending=False)


,product_line,abs_error
2,urban,32.125344
1,mountain,27.149373
0,electric,17.046806


In [197]:
df.groupby(['region'])['abs_error'].mean().reset_index().sort_values(by='abs_error', ascending=False)

,region,abs_error
1,inland,28.592618
2,metro,27.284027
0,coast,20.444878


In [198]:
df.groupby(['product_line', 'channel'])['abs_error'].mean().reset_index().sort_values(by='abs_error', ascending=False)


,product_line,channel,abs_error
4,urban,ecommerce,34.223140
5,urban,store,30.027548
2,mountain,ecommerce,29.520049
3,mountain,store,24.778696
0,electric,ecommerce,18.776808
1,electric,store,15.316804


In [199]:
df.groupby(['channel', 'campaign'])['abs_error'].mean().reset_index().sort_values(by='abs_error', ascending=False)


,channel,campaign,abs_error
9,store,summer,42.879654
4,ecommerce,summer,39.998217
3,ecommerce,spring,36.876389
8,store,spring,31.917803
2,ecommerce,none,26.236636
1,ecommerce,christmas,24.850730
0,ecommerce,black_friday,23.449337
7,store,none,21.478074
6,store,christmas,21.281194
5,store,black_friday,20.395406


In [200]:
df.groupby(['region', 'channel'])['abs_error'].mean().reset_index().sort_values(by='abs_error', ascending=False)


,region,channel,abs_error
2,inland,ecommerce,30.896286
4,metro,ecommerce,29.519539
3,inland,store,26.288950
5,metro,store,25.048515
0,coast,ecommerce,22.104173
1,coast,store,18.785583


In [201]:
df.groupby(['region', 'channel', 'campaign'])['abs_error'].mean().reset_index().sort_values(by='abs_error', ascending=False)


,region,channel,campaign,abs_error
24,metro,ecommerce,summer,72.029385
29,metro,store,summer,60.896270
23,metro,ecommerce,spring,60.847567
28,metro,store,spring,49.886869
9,coast,store,summer,40.524892
10,inland,ecommerce,black_friday,39.197980
3,coast,ecommerce,spring,39.033460
4,coast,ecommerce,summer,37.244108
11,inland,ecommerce,christmas,35.385004
16,inland,store,christmas,34.420455


In [ ]:
# també podem comprovar si és cert que l'error sembla ser major en els mesos més recents
# per exemple, calculem l'error només a partir de 2025
df['year_month'] = df['month'].str[:7]
df['year'] = df['month'].str[:4]
df['month'] = df['month'].str[5:]


df[df['year'] >= '2025'].groupby('year')['abs_error'].mean().reset_index().sort_values(by='abs_error', ascending=False)

,year,abs_error
1,2026,30.946596
0,2025,25.879536


In [218]:
# i ara l'anterior a 2025
df[df['year'] < '2025'].groupby('year')['abs_error'].mean().reset_index().sort_values(by='abs_error', ascending=False)

,year,abs_error
0,2021,25.856341
1,2022,24.464553
2,2023,24.285494
3,2024,23.963571


In [ ]:
# fins i tot, si finalment heu separat l'any i el mes en columnes diferents a month, podeu calcular
# la mitjana de l'error per any o per mes

df[df['month'] == '12'].groupby('year')['abs_error'].mean().reset_index().sort_values(by='abs_error', ascending=False)

,year,abs_error
2,2023,26.310325
1,2022,23.792929
0,2021,22.892256
3,2024,22.222783
4,2025,22.109989


#### Fer prediccions i testejar-les


Anem a fer prediccions com cal, testejant amb dades que no apareixen en l'entrenament de la baseline.

Per exemple, fem les prediccions amb les dades de 2021-2024, i utilitzem les de 2025 per validar la baseline.

Podeu fer-ho per qualsevol agrupació. En els comentaris següents la suggerència és per prediccions per famílies.

Seria convenient haver fet abans la separació de la columna month en una columna year_solt i una altra month_solt tal com hem vist a classe.

In [226]:
# 1. Vendes totals de cada família en cada mes

# 2. Separar l'historial disponible del període que volem predir
# 2.1 un dataframe per a les dades de 2021-2024
# 2.2 un altre dataframe per a les dades de 2025
# 3. Mitjana mensual històrica de cada família en el dataframe de 2021-2024. Guardem el resultat
# en una columna nova
# 4. Afegim eixa columna nova (els mesos de prova) al dataframe de 2025
# 5. Mostrem un llistat del dataframe de 2025 (per exemple les primeres 25 files) on aparega:
# - l'any (serà 2025)
# - el mes (columna month_solt)
# - la família (product_line)
# - les vendes reals (sales_units)
# - la predicció de vendes per a eixa família

In [227]:
# calculem la mitjana de l'error absolut (valor absolut de la diferència entre valor real i predicció)

# el mostrem

# si el valor és alt pot ser que el model funciona mal, però també pot ser que els valors de les vendes 
# siguen alts

# per això és millor treure l'error relatiu 
# (recordeu que estem en el dataframe de 2025)
# ho podem fer simplement sumant la columna sales_units per una banda
# sumant d'una altra banda els errors absoluts (valor absolut de vendes reals - predicció)
# calculant quin percentatge suposa la suma dels errors absoluts sobre el total de vendes
# (una senzilla regla de tres: 100 * suma_errors / suma_vendes)


Hem de tenir en compte que estem calculant errors absoluts. És a dir, la desviació de la predicció respecte a la quantitat real. Això implica que quan parlem de quantitats grans, la desviació absoluta també serà més gran. Per exemple, pot semblar que una desviació de 100 unitats és major que una de 5, però si la desviació de 100 unitats és respecte a quantitats de cinc xifres sempre serà molt millor que una de 5 respecte a quantitats de 2 xifres. Per això una de les coses que podríem fer a partir d'ací és tornar a fer tots els càlculs però en lloc de mostrar l'error en valor absolut, fer-ho amb el percentatge respecte al valor real, per exemple.

A continuació hi ha un exemple si cal explicar com fer-ho, després podem deixar que l'alumnat revise totes les proves que ha fet mostrant els errors en valor absolut i també en percentatge.

Podeu repetir l'exercici tantes vegades com tingueu ganes, amb agrupacions i periodes diferents. Per exemple, utilitzant la mitjana només de 2022-2024 en lloc de 2021-2024, o agrupar per familia i mes, per campanya...